# Tratativa das matches
## Etapa tratativa dos dados e gravação na silver

Esta etapa consiste em tratar os dados das partidas, renomear colunas, alterar datatypes e selecionar dados relevantes para o processo.

In [0]:
from pyspark.sql.functions import *
from datetime import datetime, timedelta

In [0]:
df_silver = spark.read.table("apifootball.silver.matches") #não existe na primeira execução
max_dh_bronze = (
    spark.read.table("apifootball.bronze.matches_raw")
    .agg(max("dh_processing_br").alias("max_dh"))
    .collect()[0]["max_dh"]
)
df_bronze = (
    spark.read.table("apifootball.bronze.matches_raw").alias("r")
    .filter(col("r.match_status") == "Finished")
    .filter(col("r.dh_processing_br") == max_dh_bronze)
    .join(df_silver.alias("s"), col("r.match_id") == col("s.match_id"), "leftanti") # segunda rodada +, primeira comentar
)#167

In [0]:
df_tratativa = (
    df_bronze.alias("b")
    .select(
        col("b.match_id").cast("int").alias("match_id"),
        to_timestamp(concat_ws(" ", "match_date", "match_time"),"yyyy-MM-dd HH:mm").alias("match_datetime"),
        col("b.match_date").cast("date").alias("match_date"),
        col("b.match_round"),
        col("b.match_status"),
        col("b.match_hometeam_id").cast("int").alias("hometeam_id"),
        col("b.match_hometeam_name").alias("hometeam_name"),
        col("b.match_awayteam_id").cast("int").alias("awayteam_id"),
        col("b.match_awayteam_name").alias("awayteam_name"),
        col("b.match_hometeam_score").cast("int").alias("hometeam_score"),
        col("b.match_awayteam_score").cast("int").alias("awayteam_score"),
        col("b.match_stadium").alias("stadium"),
        col("b.match_referee").alias("referee"),
        col("b.league_name"),        
        col("b.league_year").cast("int").alias("league_year"),
        expr("current_timestamp() - INTERVAL 3 HOURS").alias("datetime_processing_br"),
        col("b.dh_processing_br").alias("datetime_processing_bronze_br")
    )
)


In [0]:
df_indicadores = (
    df_tratativa
    .select(
        col("match_id"),
        col("match_datetime"),
        col("match_date"),
        col("match_round"),
        col("match_status"),
        col("hometeam_id"),
        col("hometeam_name"),
        col("awayteam_id"),
        col("awayteam_name"),
        col("hometeam_score"),
        col("awayteam_score"),
        when(col("hometeam_score") > col("awayteam_score"), col("hometeam_name")) \
        .when(col("awayteam_score") > col("hometeam_score"), col("awayteam_name")) \
        .otherwise(lit(None)).alias("winner"),
        when(col("hometeam_score") == col("awayteam_score"), lit(1)).otherwise(lit(0)).alias("is_draw"),
        (col("hometeam_score") + col("awayteam_score")).alias("total_goals"),
        abs(col("hometeam_score") - col("awayteam_score")).alias("goal_difference"),
        col("stadium"),
        col("referee"),
        col("league_name"),
        col("league_year"), 
        col("datetime_processing_br"),
        col("datetime_processing_bronze_br")
    )
).orderBy(col("match_datetime").asc())

In [0]:
df_indicadores.write.mode("append").saveAsTable("apifootball.silver.matches")

In [0]:
#spark.read.table("apifootball.silver.matches").display()